In [2]:
import os
import pandas as pd

base = '/content' if os.path.exists('/content/olist_orders_dataset.csv') else '/'
print("Base folder:", base)

orders    = pd.read_csv(f'{base}/olist_orders_dataset.csv',
                        parse_dates=['order_purchase_timestamp',
                                     'order_delivered_customer_date',
                                     'order_estimated_delivery_date'])
reviews   = pd.read_csv(f'{base}/olist_order_reviews_dataset.csv')
items     = pd.read_csv(f'{base}/olist_order_items_dataset.csv')
customers = pd.read_csv(f'{base}/olist_customers_dataset.csv')

for n, d in [('orders', orders), ('reviews', reviews), ('items', items), ('customers', customers)]:
    print(n, d.shape)

Base folder: /content
orders (99441, 8)
reviews (99224, 7)
items (112650, 7)
customers (99441, 5)


In [3]:
import numpy as np
import pandas as pd

cutoff  = pd.Timestamp('2018-03-01')
horizon = cutoff + pd.Timedelta(days=90)

# Orders with customer_unique_id, delivered only
o = orders.merge(customers[['customer_id', 'customer_unique_id']], on='customer_id')
o = o[o['order_status'] == 'delivered'].copy()

# TARGET: customers who ordered again within 90 days after the cutoff
future_ids = set(o[(o['order_purchase_timestamp'] >= cutoff) &
                   (o['order_purchase_timestamp'] <  horizon)]['customer_unique_id'])

# PAST: orders purchased AND delivered before the cutoff
past = o[(o['order_purchase_timestamp'] < cutoff) &
         (o['order_delivered_customer_date'] < cutoff)].copy()

print("Past orders:", len(past), "| Future customers:", len(future_ids))

Past orders: 53644 | Future customers: 19988


In [4]:
# Delivery features
past['delay_days']    = (past['order_delivered_customer_date'] - past['order_estimated_delivery_date']).dt.days  # >0 = late
past['delivery_days'] = (past['order_delivered_customer_date'] - past['order_purchase_timestamp']).dt.days

# Order value and freight
it = items.groupby('order_id').agg(price=('price', 'sum'), freight=('freight_value', 'sum')).reset_index()
past = past.merge(it, on='order_id', how='left')

# Review score (only reviews created before the cutoff)
rv = reviews[pd.to_datetime(reviews['review_creation_date']) < cutoff]
rv = rv.groupby('order_id')['review_score'].mean().reset_index()
past = past.merge(rv, on='order_id', how='left')

# One row per customer
feat = past.groupby('customer_unique_id').agg(
    recency=('order_purchase_timestamp', lambda x: (cutoff - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('price', 'sum'),
    freight_sum=('freight', 'sum'),
    avg_delay_days=('delay_days', 'mean'),
    late_share=('delay_days', lambda x: (x > 0).mean()),
    avg_delivery_days=('delivery_days', 'mean'),
    avg_review=('review_score', 'mean'),
).reset_index()

feat['freight_ratio'] = feat['freight_sum'] / feat['monetary'].replace(0, np.nan)
feat['monetary'] = np.log1p(feat['monetary'])
feat['returned'] = feat['customer_unique_id'].isin(future_ids).astype(int)

print("Customers:", len(feat), "| Returned:", feat['returned'].sum(),
      f"({feat['returned'].mean()*100:.2f}%)")
feat[['avg_delay_days', 'late_share', 'avg_delivery_days', 'avg_review', 'freight_ratio']].describe().round(2)

Customers: 52013 | Returned: 349 (0.67%)


,avg_delay_days,late_share,avg_delivery_days,avg_review,freight_ratio
count,52013.00,52013.00,52013.00,51349.00,52013.00
mean,-12.71,0.05,12.46,4.18,0.30
std,9.46,0.22,9.08,1.25,0.29
min,-140.00,0.00,0.00,1.00,0.00
25%,-17.00,0.00,7.00,4.00,0.13
50%,-13.00,0.00,11.00,5.00,0.22
75%,-9.00,0.00,16.00,5.00,0.37
max,181.00,1.00,209.00,5.00,5.74


In [5]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

sets = {
    'RFM only': ['recency', 'frequency', 'monetary'],
    'RFM + delivery + reviews': ['recency', 'frequency', 'monetary', 'avg_delay_days',
                                 'late_share', 'avg_delivery_days', 'avg_review', 'freight_ratio'],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, cols in sets.items():
    model = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                          LogisticRegression(class_weight='balanced', max_iter=1000))
    res = cross_validate(model, feat[cols], feat['returned'], cv=cv,
                         scoring=['roc_auc', 'average_precision'])
    print(f"\n{name}")
    print(f"  ROC-AUC: {res['test_roc_auc'].mean():.3f} (+/- {res['test_roc_auc'].std():.3f})")
    print(f"  PR-AUC : {res['test_average_precision'].mean():.3f} (+/- {res['test_average_precision'].std():.3f})")


RFM only
  ROC-AUC: 0.602 (+/- 0.019)
  PR-AUC : 0.026 (+/- 0.012)

RFM + delivery + reviews
  ROC-AUC: 0.605 (+/- 0.012)
  PR-AUC : 0.025 (+/- 0.011)


## Result: delivery delay and review score do not improve repurchase prediction

Adding average delivery delay, share of late orders, delivery time, review score and
freight ratio to RFM features did not improve the model (5-fold CV):

| Features | ROC-AUC | PR-AUC |
|---|---|---|
| RFM only | 0.602 | 0.026 |
| RFM + delivery + reviews | 0.605 | 0.025 |

The difference is within the fold-to-fold variation. Only about 5% of orders were
late and just 349 customers repurchased, so weak effects are hard to detect.
**Takeaway:** on this dataset, repeat purchase is not explained by delivery
experience; retention efforts should focus on other levers (offers to
high-value one-timers, see notebook 06).